# Phase 8 — data cleaning notebook

Walks through preparing the collected laptop-price dataset and writes:

- `data/processed/laptop_prices_clean.csv`
- `reports/cleaning_report.md` / `.json`

It does **not** create any data. Point `RAW_PATH` at the file you collected.
See `docs/data_collection_plan.md` and `docs/data_dictionary.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from luct_ai import cleaning

# Point this at the CSV you actually collected (not the empty template).
RAW_PATH = ROOT / 'data' / 'raw' / 'laptop_prices_raw.csv'
RATES_PATH = ROOT / 'data' / 'fx_rates.csv'
RAW_PATH

In [ ]:
raw = cleaning.read_dataset(RAW_PATH)
print('records:', len(raw))
raw.head()

## 2. Missing values before cleaning

In [ ]:
cleaning.summarise_missingness(raw)

## 3. Standardise, flag, convert, de-duplicate and split

`clean_dataset` runs the whole Phase 8 pipeline and returns a report object.

In [ ]:
rates = cleaning.load_fx_rates(RATES_PATH)
result = cleaning.clean_dataset(raw, rates=rates)

for note in result.notes:
    print('-', note)

## 4. Duplicates removed

In [ ]:
result.duplicates_removed

## 5. Quality flags (impossible specs, unmapped values, outliers)

In [ ]:
result.flag_counts

## 6. Missing values after cleaning and the grouped split

In [ ]:
display(result.missingness_after)
display(result.split_summary)
display(result.currency_summary)

## 7. Sanity check: no laptop group leaks across splits

In [ ]:
frame = result.frame
keys = (frame[['brand', 'model', 'processor', 'ram_gb', 'storage_gb', 'storage_type']]
        .astype('string').fillna('NA').agg('|'.join, axis=1))
checks = frame.assign(key=keys).groupby('key')['split'].nunique()
assert checks.max() == 1, 'a laptop group leaked across splits!'
print('OK — every group appears in exactly one split')

## 8. Write the cleaned dataset and the report

In [ ]:
cleaning.write_dataset(frame, ROOT / 'data' / 'processed' / 'laptop_prices_clean.csv')
result.write_report(ROOT / 'reports' / 'cleaning_report.json')
print('written:', len(frame), 'records')